# 02 · Previsão mensal por variável
    Execute o notebook 01 primeiro. A análise usa previsões diretas por variável e horizonte
    com defasagens mensais. Compare o erro por horizonte e amplie a janela de validação
    antes de publicar resultados do estudo. O pickle é interno e deve ser carregado
    somente quando produzido pela própria pipeline, sob as mesmas versões das bibliotecas.

## Configurar caminhos

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
TERRA = ROOT / 'data/terra'
RUN = TERRA / 'analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Definir alvos, horizontes e amostra de treino

O ajuste e a validação cronológica usam uma amostra reproduzível de locais.
A inferência usa todos os locais com defasagens suficientes no mês de corte.

In [ ]:
import json
import pickle

from feynmans_legacy.analog_study import LAGS, publish_forecasts, train_forecasts

HORIZONTES = [1, 2, 3]
LOCAIS_TREINO = 400
SEMENTE = 42
selection = json.loads((RUN / "selecao.json").read_text(encoding="utf-8"))
TARGETS = selection["alvos_previsao"]
terra_path = TERRA / "terra.parquet"
locations = pd.read_parquet(RUN / "coordenadas.parquet", columns=["location_id"])
sample = locations["location_id"].sort_values().sample(
    n=min(LOCAIS_TREINO, len(locations)), random_state=SEMENTE
)
history = pd.read_parquet(
    terra_path, columns=["location_id", "anomes", *TARGETS],
    filters=[("location_id", "in", sample.tolist())],
)
sample_size = history["location_id"].nunique()
print(f"Treino com {sample_size:,} locais e {len(TARGETS)} variáveis dinâmicas")

## Treinar e comparar os métodos

In [ ]:
bundle, metrics = train_forecasts(history, TARGETS, HORIZONTES)
metrics["locais_amostra"] = history["location_id"].nunique()
metrics["semente_amostra"] = SEMENTE
if not bundle["modelos"]:
    raise ValueError("Histórico insuficiente para publicar as previsões configuradas.")

## Salvar o modelo e as métricas

In [ ]:
model_dir = RUN / "modelos"
model_dir.mkdir(exist_ok=True)
artifact = model_dir / "previsao_mensal.pkl"
with artifact.open("wb") as file:
    pickle.dump(bundle, file, protocol=pickle.HIGHEST_PROTOCOL)
metrics.to_parquet(RUN / "metricas_previsao.parquet", index=False)


## Conferir o erro por horizonte

In [ ]:
print(metrics.to_string(index=False))

A inferência em lote lê apenas os meses necessários no Parquet observado.
Ao atualizar o mês de corte, regenere o artefato e revise as métricas.
O mapa lê o Parquet publicado.

## Recarregar o modelo e publicar as previsões

In [ ]:
cutoff = pd.Period(bundle["origem"], freq="M")
lag_months = [int((cutoff - lag).strftime("%Y%m")) for lag in LAGS]
recent = pd.read_parquet(
    terra_path, columns=["location_id", "anomes", *TARGETS],
    filters=[("anomes", "in", lag_months)],
)
forecasts = publish_forecasts(recent, artifact)
if forecasts.empty:
    raise ValueError("Nenhum local possui as entradas necessárias no mês de corte.")
forecasts.to_parquet(RUN / "previsoes.parquet", index=False)
print(f"{len(forecasts):,} previsões para {forecasts['location_id'].nunique():,} locais")

## Conferir os valores previstos

In [ ]:
print(forecasts.head().to_string(index=False))